# 🚀 02: T4 Controlled-Retrieval Baseline-Policy Sweeps

*Part of the KV-Cache Eviction Capstone Series*
*Estimated time: 45–180 minutes (if executed)*

---

Welcome to the first evaluation phase. This notebook is designed for a standard NVIDIA T4 GPU. It uses the canonical controlled retrieval corpus at strictly matched cache budgets.

For the baseline development sweep, the notebook selects 25 deterministic non-test records from each planned length stratum, for 100 records total. The 40 held-out test records are reserved for Notebook 04.

### Experiment Roadmap
```text
[Controlled rows] → [Short attention profile] → [Matched-budget sweep] → [Observed CSV]
```


## Step 1: Why Does This Matter?

Before we try to invent a learned eviction policy, we need to know exactly how well the existing heuristics perform. If a simple "keep the first 4 tokens and the most recent 124 tokens" rule solves the problem, we do not need a complex model.

**Runtime Contract:**
The default T4 profile targets the 1.5B model in 4-bit mode. If your GPU runs out of memory, the code will catch the error and record a **requires larger GPU** status. It will never fabricate a measurement.


In [ ]:
NOTEBOOK_ID = '02_t4_ruler_baseline_sweeps'
REQUESTED_PROFILE = 't4'

# Colab bootstrap: install pinned dependencies and unpack the shared core.
# Upload kvcore_bundle.zip supplied with this notebook suite if kvcore is not present.
from pathlib import Path
import sys, subprocess, zipfile

PINNED = [
    'transformers==4.56.2', 'accelerate==1.10.1', 'datasets==4.0.0',
    'huggingface_hub==0.34.4', 'bitsandbytes==0.47.0', 'safetensors==0.6.2',
    'sentencepiece==0.2.1', 'scipy==1.16.1', 'matplotlib==3.10.6',
    'seaborn==0.13.2', 'pandas==2.3.2',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *PINNED])

if not Path('kvcore').exists():
    try:
        from google.colab import files
        print('Upload kvcore_bundle.zip from the delivered suite.')
        uploaded = files.upload()
        archive = next((Path(name) for name in uploaded if name.endswith('.zip')), None)
        if archive is None:
            raise FileNotFoundError('Please upload kvcore_bundle.zip.')
        with zipfile.ZipFile(archive) as zf:
            zf.extractall('.')
    except ImportError as error:
        raise RuntimeError('Run in Google Colab or place the kvcore directory beside this notebook.') from error

sys.path.insert(0, str(Path('.').resolve()))
from kvcore import *
from kvcore.config import BENCHMARKS, MODELS, POLICY_DEFAULTS, PROFILES, SUITE_VERSION
print({'suite_version': SUITE_VERSION, 'ruler_revision': BENCHMARKS['ruler']['revision'], 'longbench_revision': BENCHMARKS['longbench']['revision']})


In [ ]:
import json, os, platform, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

REQUESTED_PROFILE = REQUESTED_PROFILE  # defined by the notebook title cell
NOTEBOOK_ID = globals().get('NOTEBOOK_ID', 'runtime')
set_all_seeds(590)
profile, runtime_status = select_profile(REQUESTED_PROFILE)
run_root = ensure_run_root(f'kv_eviction_{NOTEBOOK_ID}')
manifest = run_manifest(
    run_root,
    notebook=NOTEBOOK_ID,
    requested_profile=REQUESTED_PROFILE,
    active_profile=profile.name,
    model=model_spec(profile.model_tier),
    runtime_status=runtime_status,
)
print(json.dumps({'notebook': NOTEBOOK_ID, 'requested_profile': REQUESTED_PROFILE, 'active_profile': profile.name, 'runtime_status': runtime_status, 'run_root': str(run_root)}, indent=2))
if not torch.cuda.is_available():
    raise RuntimeError('A CUDA GPU runtime is required for model execution. In Colab: Runtime > Change runtime type > GPU.')
print('GPU:', torch.cuda.get_device_name(0), 'VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))


In [ ]:
# Model and tokenizer are always loaded at immutable Hub revisions from kvcore.config.
# If the runtime is smaller than the requested tier, runtime_status records the fallback.
model, tokenizer = load_model_and_tokenizer(model_spec(profile.model_tier), load_mode=profile.load_mode, attn_implementation='sdpa')
print({'model': model.config._name_or_path, 'requested_profile': REQUESTED_PROFILE, 'active_profile': profile.name, 'load_mode': profile.load_mode})


In [ ]:
# All policies use the same per-example cache budget. Full cache is the uncompressed reference.
POLICY_SPECS = [
    {'type': 'full'}, {'type': 'fifo', 'min_recent': 32},
    {'type': 'random', 'seed': 590, 'min_recent': 32}, {'type': 'uniform', 'min_recent': 32},
    {'type': 'sink_recent', 'sink_tokens': 4, 'min_recent': 32},
    {'type': 'attention_topk', 'sink_tokens': 4, 'min_recent': 32},
    {'type': 'h2o', 'recent_fraction': 0.50, 'sink_tokens': 4, 'min_recent': 32},
]


In [ ]:
RUN_T4_SWEEP = False  # Set True only after checking GPU memory, disk, and desired allocation.
all_controlled_rows = load_controlled_retrieval(tokenizer=tokenizer, seed=590)
corpus_audit = audit_controlled_retrieval_corpus(all_controlled_rows)
assert corpus_audit['passed'], corpus_audit
# 25 deterministic train/validation rows per length = 100 baseline-development records; held-out test remains unused.
rows = load_controlled_retrieval(tokenizer=tokenizer, seed=590, max_examples=25, partitions=('train', 'validation'))
assert len(rows) == 100 and all(row['partition'] in {'train', 'validation'} for row in rows)
build_benchmark_manifest(rows, run_root / 'manifests' / 't4_controlled_baseline_rows.json', benchmark_name='t4_controlled_baseline', seed=590)
print(pd.DataFrame(rows).groupby(['planned_context_length','partition']).size())


## Step 2: Two-Pass Attention Protocol

Extracting attention matrices from every layer for every token is incredibly memory-intensive. To survive on a T4, we use a two-pass discipline: a short, eager pass collects attention statistics for the Top-K policies, and the main evaluation pass uses memory-efficient SDPA without returning attention weights.


In [ ]:
profile_row = rows[0]
profile_ids = tokenise_prompt(tokenizer, profile_row['prompt'], min(profile.attention_profile_context_cap, profile.max_context_tokens), next(model.parameters()).device)
profile_meta = profile_short_context(model, profile_ids, max_tokens=min(profile_ids.shape[-1], profile.attention_profile_context_cap), run_root=run_root, tag='t4_short_profile', prefill_chunk_tokens=profile.prefill_chunk_tokens)
attention_profile = load_profile(profile_meta['profile_path'], device=next(model.parameters()).device) if profile_meta['attention_available'] else None
write_json(run_root / 'attention_profiles' / 't4_profile_protocol.json', profile_meta)
print(profile_meta)


## Step 3: Matched-Budget Baseline Sweep

This is the main event. We will evaluate every policy using the exact same budget values to ensure a fair comparison. 

**Before You Run:**
Set `RUN_T4_SWEEP = True` only if you have confirmed your GPU has sufficient memory (at least 15GB free). Otherwise, leave it `False` to record a safe "not run" status.


In [ ]:
if RUN_T4_SWEEP:
    records = evaluate_rows(
        model, tokenizer, rows, POLICY_SPECS, profile.budgets, profile.max_context_tokens,
        profile.decode_tokens, profile.prefill_chunk_tokens, run_root, profile=attention_profile,
        artifact_stem='t4_controlled_retrieval_baselines',
    )
else:
    records = [requires_gpu_record('T4 or larger', 'Set RUN_T4_SWEEP=True', 'Evaluation deliberately disabled until user confirms runtime allocation.')]
pd.DataFrame(records).to_csv(run_root / 'results' / 't4_controlled_retrieval_baselines.csv', index=False)
display(pd.DataFrame(records).head())


## Step 4: Results-Ready Visualization

Let us visualize the quality-versus-budget trade-off for the heuristics.

### What to Look For
The plot is generated *only* from successfully observed rows. If your run recorded hardware limits, those points are safely excluded from the curves but preserved in the CSV artifact.

**Next step:** Save the `run_root` directory. We will aggregate these baseline results in Notebook 05.


In [ ]:
df = pd.read_csv(run_root / 'results' / 't4_controlled_retrieval_baselines.csv')
observed = df[df.get('status', pd.Series(dtype=str)).eq('observed')] if 'status' in df else pd.DataFrame()
if not observed.empty:
    plt.figure(figsize=(9,4)); sns.lineplot(data=observed, x='budget', y='substring_match', hue='policy', marker='o'); plt.title('Observed controlled-retrieval quality proxy versus cache budget — T4'); plt.tight_layout(); plt.savefig(run_root / 'figures' / 't4_quality_budget.png', dpi=180); plt.show()
else:
    print('No observed sweep rows yet; see the explicit status table instead of a fabricated curve.')
